# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/himanshu-yadav-10/FlyRank-ML-starter-template/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This notebook builds the transparent rule-based baseline that the Week-5 model must beat.
We follow the session's approach: **check signals first, encode a hand-written rule, review the top ten**.

**Structure:**
1. Signal audits (two bucket tables with n, one verdict each)
2. Rule encoding (score, reason code, action label) + ranked queue → CSV
3. Top-10 review (action, why it's there, what would make it wrong)
4. Weak picks + leakage check
5. Self-check

In [1]:
import os
import sys
import subprocess
import numpy as np
import pandas as pd
from pathlib import Path

# ---------- repo root ----------
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.isdir("Flyrank-ML-starter-template"):
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/himanshu-yadav-10/Flyrank-ML-starter-template", "Flyrank-ML-starter-template"],
            check=True,
        )
    os.chdir("Flyrank-ML-starter-template")
else:
    # Locate the repo root (the folder holding data/raw/) no matter where
    # the kernel started: walk up, then scan one level of subdirectories.
    def _find_repo_root(start: str):
        here = Path(start).resolve()
        for _ in range(10):
            if (here / "data" / "raw" / "content_refresh_anonymized.csv").exists():
                return here
            if here.parent == here:
                break
            here = here.parent
        for sub in Path(start).resolve().iterdir():
            if sub.is_dir() and (sub / "data" / "raw" / "content_refresh_anonymized.csv").exists():
                return sub
        return None

REPO = _find_repo_root(os.getcwd())
assert REPO is not None, "Starter CSV not found — run this notebook from the repo tree."

DATA_PATH = REPO / "data" / "processed" / "refresh_feature_vector.csv"
RAW_PATH = REPO / "data" / "raw" / "content_refresh_anonymized.csv"
OUT_DIR = REPO / "work" / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Working dir:", REPO)

# ---------- load ----------
# Prefer the prepared feature vector (target pre-computed, blanks filled).
# Fall back to raw CSV if the processed file is missing.
if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
    print(f"Loaded processed feature vector: {len(df):,} rows × {len(df.columns)} columns")
elif RAW_PATH.exists():
    df = pd.read_csv(RAW_PATH)
    num_cols = [
        "impressions_90d", "clicks_90d", "sessions_90d", "content_age_days",
        "days_since_last_update", "ctr", "avg_position", "engagement_rate",
        "scroll_rate", "ai_traffic_pct", "word_count",
    ]
    for c in num_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)
    df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
    df = df.drop_duplicates(subset=["content_id"]).reset_index(drop=True)
    df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
    df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
    print(f"Loaded raw CSV and prepped: {len(df):,} rows × {len(df.columns)} columns")
else:
    raise FileNotFoundError(f"Data not found at {DATA_PATH} or {RAW_PATH}")

BASE_RATE = df["is_declining_label"].mean()
print(f"Base rate (declining): {BASE_RATE:.3f} ({df['is_declining_label'].sum():,} / {len(df):,})")

Working dir: C:\Users\Himanshu Yadav\Desktop\Flyrank\Flyrank-ML-starter-template
Loaded processed feature vector: 30,000 rows × 52 columns
Base rate (declining): 0.542 (16,262 / 30,000)


---
## 1. Signal checks

*Two signals, each with a bucket table (visible n) and a verdict. Both are linked to real FlyRank flags: staleness sits behind the refresh flags, CTR sits behind the CTR-fix logic.*

### Signal 1 — Staleness (flag-linked: behind FlyRank's refresh flags)

**Claim:** Pages that haven't been updated in a long time decline more often.

FlyRank's refresh flags treat staleness as a core input. If the monotonic story doesn't hold, the flag needs a volume floor — which is exactly the kind of thing a signal audit exists to catch.

In [2]:
# --- Signal 1: Staleness ---
# Bucket by freshness_tier (from days_since_last_update), print n + declining rate.

staleness = (
    df.groupby("freshness_tier", observed=True)
    .agg(
        n=("is_declining_label", "size"),
        declining_n=("is_declining_label", "sum"),
        declining_rate=("is_declining_label", "mean"),
        median_days=("days_since_last_update", "median"),
        median_impressions=("impressions_90d", "median"),
    )
    .sort_values("median_days")
)

print("=== Signal 1: Staleness × Declining Rate ===")
print(f"Total rows: {len(df):,}")
print(staleness.to_string(
    formatters={
        "declining_rate": "{:.3f}".format,
        "median_days": "{:.0f}".format,
        "median_impressions": "{:.0f}".format,
    }
))

# --- The volume-floor reveal: does staleness START working once there is traffic to lose? ---
stale = df[df["days_since_last_update"] >= 180]
stale_visible = stale[stale["impressions_90d"] >= 500]
stale_lowvol = stale[stale["impressions_90d"] < 500]

print()
print("=== Volume-floor split inside the stale bucket (≥180 days) ===")
print(f"Stale, visible (impressions ≥ 500):    n={len(stale_visible):>5}   declining rate = {stale_visible['is_declining_label'].mean():.3f}")
print(f"Stale, low volume (impressions < 500): n={len(stale_lowvol):>5}   declining rate = {stale_lowvol['is_declining_label'].mean():.3f}")

=== Signal 1: Staleness × Declining Rate ===


Total rows: 30,000
                    n  declining_n declining_rate median_days median_impressions
freshness_tier                                                                  
0-30            20480        10473          0.511          20                470
31-90             175          103          0.589          41                510
91-180           9171         5604          0.611         104               1692
181+              174           82          0.471         211                 16

=== Volume-floor split inside the stale bucket (≥180 days) ===
Stale, visible (impressions ≥ 500):    n=   17   declining rate = 0.941
Stale, low volume (impressions < 500): n=  157   declining rate = 0.420


**Verdict: MIXED.**

The monotonic story is FALSE: declining rate rises from 0.511 (0-30d) to 0.611 (91-180d), then *falls* to 0.471 for the most stale bucket (181+). The reason is volume: the 181+ bucket has median ~16 impressions/90d — there's no traffic left to lose, so it isn't 'declining'.

**This saved my rule.** A plain 'stale → refresh' rule would have fired on exactly the wrong pages (dead traffic). Once you add a **visibility floor** (≥500 impressions), the signal snaps into focus: stale + visible pages decline 94% of the time vs 42% for stale-but-low-volume pages. My rule below encodes that floor.

### Signal 2 — CTR (flag-linked: behind FlyRank's CTR-fix logic)

**Claim:** Pages with impressions but low CTR are underperforming — a sign the title/snippet needs work. FlyRank's CTR-fix logic flags visible pages whose position doesn't convert to clicks.

In [3]:
# --- Signal 2: CTR quartiles among visible pages ---
# Filter to pages with volume AND valid position data (avg_position 0 = missing).
visible = df[(df["impressions_90d"] >= 500) & (df["avg_position"] > 0)].copy()

visible["ctr_quartile"] = pd.qcut(
    visible["ctr"].clip(lower=0.01),
    q=4,
    labels=["Q1_lowest_ctr", "Q2", "Q3", "Q4_highest_ctr"],
    duplicates="drop",
)

ctr_table = (
    visible.groupby("ctr_quartile", observed=True)
    .agg(
        n=("is_declining_label", "size"),
        declining_n=("is_declining_label", "sum"),
        declining_rate=("is_declining_label", "mean"),
        median_ctr=("ctr", "median"),
        median_position=("avg_position", "median"),
    )
)

print(f"=== Signal 2: CTR Quartile × Declining Rate (visible pages) ===")
print(f"Visible pages (impressions ≥ 500, position > 0): {len(visible):,} / {len(df):,}")
print(ctr_table.to_string(
    formatters={
        "declining_rate": "{:.3f}".format,
        "median_ctr": "{:.2f}".format,
        "median_position": "{:.1f}".format,
    }
))

=== Signal 2: CTR Quartile × Declining Rate (visible pages) ===
Visible pages (impressions ≥ 500, position > 0): 16,726 / 30,000
                   n  declining_n declining_rate median_ctr median_position
ctr_quartile                                                               
Q1_lowest_ctr   4550         3021          0.664       0.00            19.8
Q2              4059         2516          0.620       0.12            12.0
Q3              4065         2355          0.579       0.25             9.5
Q4_highest_ctr  4052         2069          0.511       0.56             7.8


**Verdict: CONFIRMED.**

Declining rate falls monotonically across CTR quartiles: 0.664 (lowest-CTR Q1) → 0.511 (highest-CTR Q4), with n≈4-4.5k per bucket. Low CTR correlates with decline, consistent with the CTR-fix logic.

*Caveat:* CTR and position are entangled (Q1's median position is 19.8, Q4's is 7.9). The relationship holds directionally here, but Week-5's model should check CTR within position tiers — this audit is directional, not causal.

---
## 2. Encode the rule and build the ranked queue

**The rule in plain words:** *"A page is worth refreshing if it is stale (≥180 days since last update) AND visible (≥500 impressions in the window). Among those pages, the ones with the most impressions go first."*

- **Score:** `stale_flag × visible_flag × visibility_pctrank` — a 0-1 score; 0 means "not flagged".
- **Reason code:** `stale_visible_page` (that's the one code a flagged page gets).
- **Action label:** `refresh` when flagged, `monitor` otherwise.

In [4]:
# --- Rule encoding ---

# Flags (transparent, threshold-based)
df["stale_flag"]   = (df["days_since_last_update"] >= 180).astype(int)
df["visible_flag"] = (df["impressions_90d"] >= 500).astype(int)

# Visibility: percentile rank of log-impressions (0-1 scale)
df["visibility_pctrank"] = np.log1p(df["impressions_90d"]).rank(method="average", pct=True)

# Score: flagged pages carry their visibility rank; unflagged pages get 0
df["baseline_score"] = df["stale_flag"] * df["visible_flag"] * df["visibility_pctrank"]

# One reason code per row
df["reason_code"] = np.select(
    [
        (df["stale_flag"] == 1) & (df["visible_flag"] == 1),
        (df["stale_flag"] == 1) & (df["visible_flag"] == 0),
    ],
    ["stale_visible_page", "stale_low_volume"],
    default="not_stale",
)

# One action label per row
df["action"] = np.where(df["baseline_score"] > 0, "refresh", "monitor")

# Rank by score descending
df["baseline_rank"] = df["baseline_score"].rank(method="first", ascending=False).astype(int)

print("Score distribution:")
print(f"  Flagged (score > 0, stale+visible): {(df['baseline_score'] > 0).sum():,}")
print(f"  Not flagged (score = 0):            {(df['baseline_score'] == 0).sum():,}")
print(f"  Median non-zero score: {df.loc[df['baseline_score'] > 0, 'baseline_score'].median():.4f}")
print()
print("Action distribution:")
print(df["action"].value_counts().to_string())
print()
print("Reason-code distribution:")
print(df["reason_code"].value_counts().to_string())

Score distribution:
  Flagged (score > 0, stale+visible): 17
  Not flagged (score = 0):            29,983
  Median non-zero score: 0.7784

Action distribution:
action
monitor    29983
refresh       17

Reason-code distribution:
reason_code
not_stale             29826
stale_low_volume        157
stale_visible_page       17


In [5]:
# --- Write ranked queue to CSV ---

queue = df.sort_values("baseline_rank")
queue = queue[
    [
        "baseline_rank",
        "content_id",
        "client_id",
        "baseline_score",
        "reason_code",
        "action",
        "days_since_last_update",
        "impressions_90d",
        "avg_position",
        "ctr",
        "content_age_days",
        "is_declining_label",
    ]
]

csv_path = OUT_DIR / "baseline_action_score.csv"
queue.to_csv(csv_path, index=False)
print(f"Wrote ranked queue: {csv_path}")
print(f"Queue length: {len(queue):,} rows")
print(f"Top-50 declining rate in queue: {queue.head(50)['is_declining_label'].mean():.3f}")
print(f"Base rate for comparison:       {BASE_RATE:.3f}")

Wrote ranked queue: C:\Users\Himanshu Yadav\Desktop\Flyrank\Flyrank-ML-starter-template\work\outputs\baseline_action_score.csv
Queue length: 30,000 rows
Top-50 declining rate in queue: 0.740
Base rate for comparison:       0.542


In [6]:
# --- Run receipt (metrics JSON) ---
import json

flagged_set = df[(df["stale_flag"] == 1) & (df["visible_flag"] == 1)]
receipt = {
    "task": "w04_baseline_score",
    "data_slice": "starter 30k (processed feature vector)",
    "base_rate": float(BASE_RATE),
    "signals": [
        {
            "name": "staleness",
            "verdict": "MIXED",
            "note": "monotonic staleness story fails (181+ bucket drops); holds once a >=500-impression volume floor is added — stale+visible 0.941 declining vs 0.420 stale-low-volume",
        },
        {
            "name": "ctr_vs_decline",
            "verdict": "CONFIRMED",
            "note": "declining rate falls monotonically 0.664 -> 0.511 across CTR quartiles; entangled with position, so directional not causal",
        },
    ],
    "rule": {
        "plain_words": "stale (>=180d) AND visible (>=500 impressions) are worth refreshing; most impressions go first",
        "score": "stale_flag * visible_flag * visibility_pctrank",
        "reason_code": "stale_visible_page",
        "action": "refresh if score > 0 else monitor",
        "features": ["days_since_last_update", "impressions_90d"],
    },
    "queue": {
        "rows": int(len(queue)),
        "flagged": int(flagged_set.shape[0]),
        "declining_rate_flagged": float(flagged_set["is_declining_label"].mean()),
        "declining_rate_top10": float(queue.head(10)["is_declining_label"].mean()),
        "declining_rate_top50": float(queue.head(50)["is_declining_label"].mean()),
        "output_csv": "work/outputs/baseline_action_score.csv",
    },
}

receipt_path = OUT_DIR / "baseline_metrics.json"
receipt_path.write_text(json.dumps(receipt, indent=2, sort_keys=True))
print(f"Wrote run receipt: {receipt_path}")
print(f"Receipt size: {receipt_path.stat().st_size} bytes")

Wrote run receipt: C:\Users\Himanshu Yadav\Desktop\Flyrank\Flyrank-ML-starter-template\work\outputs\baseline_metrics.json
Receipt size: 1245 bytes


---
## 3. Top-10 review

*For each of the top 10: the action, why it's there, and what would make it wrong.*

In [7]:
# --- Top-10 review ---

def what_would_make_it_wrong(row) -> str:
    """A per-page reason this pick could be a false alarm."""
    reasons = []
    if row["ctr"] == 0:
        reasons.append("ctr is 0.00 — clicks may be under-reported; the flag could be firing on a measurement gap, not a page problem")
    elif row["avg_position"] <= 12:
        reasons.append("already ranks near page one — refresh could waste effort if demand loss is the cause, not staleness")
    else:
        reasons.append("staleness + volume is right, but if impression volume is a one-off spike, the page may not deserve a human hour")
    if row["is_declining_label"] == 0:
        reasons.append("not currently declining — a refresh might be unnecessary churn")
    return " · ".join(reasons)

top10 = queue.head(10).copy()

print("=" * 100)
print("TOP-10 REVIEW")
print("=" * 100)
for _, r in top10.iterrows():
    print()
    print(
        f"[#{int(r['baseline_rank']):>2}] action={r['action']:<8} score={r['baseline_score']:.4f} "
        f"reason={r['reason_code']}  decl_label={int(r['is_declining_label'])}"
    )
    print(
        f"     why: stale {int(r['days_since_last_update'])}d, {int(r['impressions_90d']):,} impressions/90d, "
        f"pos {r['avg_position']}, ctr {r['ctr']:.2f}"
    )
    print(f"     wrong if: {what_would_make_it_wrong(r)}")

print()
print(f"Declining in top 10: {top10['is_declining_label'].sum()}/{len(top10)} = {top10['is_declining_label'].mean():.1%}")
print(f"Base rate: {BASE_RATE:.1%}")
print(f"Lift: {top10['is_declining_label'].mean() / BASE_RATE:.2f}x" if BASE_RATE > 0 else "")

TOP-10 REVIEW

[# 1] action=refresh  score=0.9866 reason=stale_visible_page  decl_label=1
     why: stale 194d, 61,678 impressions/90d, pos 19.7, ctr 0.15
     wrong if: staleness + volume is right, but if impression volume is a one-off spike, the page may not deserve a human hour

[# 2] action=refresh  score=0.9862 reason=stale_visible_page  decl_label=1
     why: stale 194d, 59,472 impressions/90d, pos 24.8, ctr 0.13
     wrong if: staleness + volume is right, but if impression volume is a one-off spike, the page may not deserve a human hour

[# 3] action=refresh  score=0.9556 reason=stale_visible_page  decl_label=1
     why: stale 194d, 25,715 impressions/90d, pos 22.2, ctr 0.23
     wrong if: staleness + volume is right, but if impression volume is a one-off spike, the page may not deserve a human hour

[# 4] action=refresh  score=0.9081 reason=stale_visible_page  decl_label=1
     why: stale 193d, 13,299 impressions/90d, pos 10.5, ctr 0.49
     wrong if: already ranks near page on

---
## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or label-derived/future-window inputs leaked in.*

In [8]:
# --- Weak pick analysis ---
# The rule's flagged set = stale AND visible. Its false positives are the ones NOT declining.

flagged = df[(df["stale_flag"] == 1) & (df["visible_flag"] == 1)]
fp = flagged[flagged["is_declining_label"] == 0]
tp = flagged[flagged["is_declining_label"] == 1]

print("Flagged set = stale AND visible (≥500 impressions):")
print(f"  Total:              {len(flagged):,}")
print(f"  Declining (TP):     {len(tp):,}  ({len(tp)/len(flagged):.1%})")
print(f"  Not declining (FP): {len(fp):,}  ({len(fp)/len(flagged):.1%})")
print()
print("False-positive profile (flagged but NOT declining):")
if len(fp):
    print(f"  median impressions:   {fp['impressions_90d'].median():,.0f}")
    print(f"  median avg_position:  {fp['avg_position'].median():.1f}")
    print(f"  median ctr:           {fp['ctr'].median():.2f}")
    print(f"  median days_since_upd: {fp['days_since_last_update'].median():.0f}")
else:
    print("  none — every flagged page is declining.")

print()
print("--- Leakage check ---")
print("Inputs the rule uses:")
print("  - days_since_last_update  (historical, backward-looking)")
print("  - impressions_90d         (historical, backward-looking)")
print()
print("Inputs the rule does NOT use:")
print("  - trend_direction  → label source, never a feature")
print("  - trend_pct        → label source, never a feature")
print("  - is_declining_label → the target — evaluation only")
print("  - health_score / any product flag → circular, never a feature")
print()
rule_features = ["days_since_last_update", "impressions_90d"]
leakage_cols = ["trend_direction", "trend_pct", "is_declining_label"]
print(f"Feature columns used: {rule_features}")
print(f"Leakage columns in the frame: {[c for c in leakage_cols if c in df.columns]} (used for evaluation only)")
print("No future-window inputs and no label-derived features — clean.")

Flagged set = stale AND visible (≥500 impressions):
  Total:              17
  Declining (TP):     16  (94.1%)
  Not declining (FP): 1  (5.9%)

False-positive profile (flagged but NOT declining):
  median impressions:   1,316
  median avg_position:  21.8
  median ctr:           0.15
  median days_since_upd: 194

--- Leakage check ---
Inputs the rule uses:
  - days_since_last_update  (historical, backward-looking)
  - impressions_90d         (historical, backward-looking)

Inputs the rule does NOT use:
  - trend_direction  → label source, never a feature
  - trend_pct        → label source, never a feature
  - is_declining_label → the target — evaluation only
  - health_score / any product flag → circular, never a feature

Feature columns used: ['days_since_last_update', 'impressions_90d']
Leakage columns in the frame: ['trend_direction', 'trend_pct', 'is_declining_label'] (used for evaluation only)
No future-window inputs and no label-derived features — clean.


---
## Self-check

- [x] Every section is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] Claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to repo under `work/notebooks/` — then submit repo URL on the card. Done.